In [1]:
# Cell 1
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

# Set this before importing torch or transformers
os.environ["CUDA_VISIBLE_DEVICES"] = "3,4,5,6,7"

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)

print("Libraries imported and GPUs set.")

/data/literature/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/data/literature/.venv/lib/python3.12/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Libraries imported and GPUs set.


In [2]:
# 📂 Dataset Paths
DATASET_DIR = Path("/home/literature/hindwi-scraper/output")
POETS_JSON = DATASET_DIR / "poets.json"

In [3]:
# 📥 Load poets.json
with open(POETS_JSON, encoding='utf-8') as f:
    poets = json.load(f)

In [4]:
# 🔄 poet_slug → poet metadata
poet_map = {poet['poet_slug']: poet for poet in poets}

In [5]:

poets_dir = DATASET_DIR / "poets"  # adjust if needed

summary = []

print("Counting poems ending with -devnagri.txt ...")

for poet_dir in tqdm([p for p in poets_dir.iterdir() if p.is_dir()]):
    poet_name = poet_dir.name
    kavita_dir = poet_dir / "kavita"

    if not kavita_dir.is_dir():
        continue

    # collect all files that end with -devnagri.txt
    devnagri_poems = [
        f for f in kavita_dir.iterdir()
        if f.is_file() and f.name.endswith("-devnagri.txt")
    ]

    summary.append({
        "poet_name": poet_name,
        "devanagari_count": len(devnagri_poems)
    })

# 📊 Per-poet DataFrame
df_summary = pd.DataFrame(summary).sort_values("devanagari_count", ascending=False)

print("\n=== Top 20 poets by Devanagari poem count ===")
print(df_summary.head(20))

# Save if needed
# df_summary.to_csv("dataset/poet_summary_filebased.csv", index=False, encoding="utf-8")


Counting poems ending with -devnagri.txt ...


100%|██████████| 2382/2382 [00:00<00:00, 8629.73it/s]


=== Top 20 poets by Devanagari poem count ===
                    poet_name  devanagari_count
1229              mona-gulati               113
339         pankaj-chaturvedi                87
1353      vijay-bahadur-singh                84
1492  krishna-murari-pahariya                81
1843             savita-singh                78
685               hari-mridul                75
178                     laltu                75
847              naveen-sagar                74
992       neelesh-raghuvanshi                71
1603      maithilisharan-gupt                70
1605  sheomangal-siddhantakar                70
1142                   malyaj                69
1746        sanjay-chaturvedi                67
507                    shubha                65
1021                 udbhrant                65
269              chandreshwar                62
694            krishna-kalpit                62
1181          manglesh-dabral                61
971             narendra-jain            

In [6]:

poets_dir = DATASET_DIR / "poets"
min_poems = 50

# 🔢 Count poems ending with "-devnagri.txt" for each poet
poet_counts = {}
for poet_dir in tqdm(list(poets_dir.iterdir()), desc="Counting poems"):
    if poet_dir.is_dir():
        kavita_dir = poet_dir / "kavita"
        if kavita_dir.is_dir():
            count = sum(1 for f in kavita_dir.glob("*-devnagri.txt"))
            if count > 0:
                poet_counts[poet_dir.name] = count

# 📊 Convert to DataFrame
poem_counts = pd.DataFrame(list(poet_counts.items()), columns=["poet_name", "devanagari_count"])

# 🎯 Filter poets with at least min_poems
poets_to_keep = set(poem_counts.loc[poem_counts["devanagari_count"] >= min_poems, "poet_name"])

print(f"Total poets with >= {min_poems} poems: {len(poets_to_keep)}")

# 📚 Build dataset (poet_name → poem text)
all_poems = []
for poet_dir in tqdm(list(poets_dir.iterdir()), desc="Loading poems"):
    if poet_dir.is_dir() and poet_dir.name in poets_to_keep:
        kavita_dir = poet_dir / "kavita"
        if kavita_dir.is_dir():
            for poem_file in kavita_dir.glob("*-devnagri.txt"):
                try:
                    poem_text = poem_file.read_text(encoding='utf-8').strip()
                    if poem_text:
                        all_poems.append({
                            "poet_name": poet_dir.name,
                            "poem": poem_text
                        })
                except Exception as e:
                    print(f"Error reading {poem_file}: {e}")

# 📑 Final DataFrame
df_filtered = pd.DataFrame(all_poems)

print(f"\nFinal dataset size: {len(df_filtered)} poems")
print(f"Unique poets: {df_filtered['poet_name'].nunique()}")
print("\nSample:")
print(df_filtered.head())


Counting poems: 100%|██████████| 2382/2382 [00:00<00:00, 19475.69it/s]


Total poets with >= 50 poems: 42


Loading poems: 100%|██████████| 2382/2382 [00:00<00:00, 38427.02it/s]


Final dataset size: 2451 poems
Unique poets: 41

Sample:
                   poet_name  \
0  acharya-ramchandra-shukla   
1  acharya-ramchandra-shukla   
2  acharya-ramchandra-shukla   
3  acharya-ramchandra-shukla   
4  acharya-ramchandra-shukla   

                                                poem  
0  भोले भाले देश भाइयों से जरा।\nभिन्न लगें, यह भ...  
1  भूलि गयो सब पूजा पाठ विचार\nकेवल उनकी ध्यान भय...  
2  तन में भरम रमायो त्यागी राज,\nसहन कठिन दुख कीन...  
3  पहले पहल पुकारा था जिसने जहाँ\nजिन नामों से जन...  
4  यद्यपि मैं हूँ एक अकेला, बैरी की सेना भारी।\nप...  


In [7]:
# Cell 15 - Update GPU configuration to use only GPU 3
import os
import torch

# Set GPU 3 only
os.environ["CUDA_VISIBLE_DEVICES"] = "4"

# Verify GPU setup
if torch.cuda.is_available():
    print(f"CUDA available: {torch.cuda.is_available()}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"Current GPU: {torch.cuda.current_device()}")
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA not available - will use CPU")

CUDA available: True
Number of GPUs: 1
Current GPU: 0
GPU name: NVIDIA RTX A6000


In [8]:
# Cell 8 - Create Numerical Labels
from sklearn.preprocessing import LabelEncoder
import numpy as np

# Create the label encoder
label_encoder = LabelEncoder()

# Fit and transform the poet names to create integer labels
df_filtered['poet_label'] = label_encoder.fit_transform(df_filtered['poet_name'])

# Store the number of classes for later use
num_classes = len(label_encoder.classes_)

print(f"✅ Labels created successfully.")
print(f"Total number of unique poets (classes): {num_classes}")
print("\nSample of the DataFrame with new 'poet_label' column:")
print(df_filtered[['poet_name', 'poet_label']].head())

✅ Labels created successfully.
Total number of unique poets (classes): 41

Sample of the DataFrame with new 'poet_label' column:
                   poet_name  poet_label
0  acharya-ramchandra-shukla           0
1  acharya-ramchandra-shukla           0
2  acharya-ramchandra-shukla           0
3  acharya-ramchandra-shukla           0
4  acharya-ramchandra-shukla           0


In [9]:
# Cell 9 - Train/Validation/Test Split (80-10-10)
from sklearn.model_selection import train_test_split

# Define features (X) and labels (y)
X = df_filtered['poem'].values
y = df_filtered['poet_label'].values

# First split: 80% train, 20% temp (which will become 10% val + 10% test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, 
    test_size=0.2,      # 20% for val+test
    random_state=42, 
    stratify=y          # Maintain class distribution across splits
)

# Second split: Split the 20% temp into 10% validation and 10% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,      # Split the 20% equally: 10% val, 10% test
    random_state=42,
    stratify=y_temp
)

# Print split statistics to verify
print("📊 Dataset Split Statistics:")
print(f"Total samples: {len(X)}")
print(f"Train samples: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Validation samples: {len(X_val)} ({len(X_val)/len(X)*100:.1f}%)")
print(f"Test samples: {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")

print("\n✅ Data split completed successfully!")

📊 Dataset Split Statistics:
Total samples: 2451
Train samples: 1960 (80.0%)
Validation samples: 245 (10.0%)
Test samples: 246 (10.0%)

✅ Data split completed successfully!


In [10]:
# Cell 10 - Convert to Hugging Face Dataset
from datasets import Dataset, DatasetDict
import pandas as pd

# Create pandas DataFrames for each split for easy conversion
train_df = pd.DataFrame({'poem': X_train, 'labels': y_train})
val_df = pd.DataFrame({'poem': X_val, 'labels': y_val})
test_df = pd.DataFrame({'poem': X_test, 'labels': y_test})

# Convert each DataFrame to a Hugging Face Dataset object
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

# Combine them all into a single DatasetDict
poet_datasets = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset,
    'test': test_dataset
})

print("📚 Hugging Face DatasetDict created:")
print(poet_datasets)

📚 Hugging Face DatasetDict created:
DatasetDict({
    train: Dataset({
        features: ['poem', 'labels'],
        num_rows: 1960
    })
    validation: Dataset({
        features: ['poem', 'labels'],
        num_rows: 245
    })
    test: Dataset({
        features: ['poem', 'labels'],
        num_rows: 246
    })
})


In [11]:
# Cell 11 - Tokenize the Datasets
from transformers import AutoTokenizer

# Define the model checkpoint
MODEL_CKPT = "google/muril-base-cased"

# Load the tokenizer associated with the model
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

# Define the function that will tokenize the 'poem' text in our dataset
def tokenize_function(examples):
    # Truncate poems longer than max_length and pad shorter ones
    return tokenizer(
        examples["poem"], 
        truncation=True, 
        padding="max_length", 
        max_length=256 # A good starting point; can be adjusted up to 512
    )

# Apply the tokenization function to all splits (train, validation, test)
print("🔤 Tokenizing datasets...")
tokenized_datasets = poet_datasets.map(tokenize_function, batched=True)

# Remove the original text column, as it's no longer needed by the model
tokenized_datasets = tokenized_datasets.remove_columns(["poem"])

# Set the format to PyTorch tensors for the Trainer
tokenized_datasets.set_format("torch")

print("\n✅ Datasets tokenized and formatted!")
print("Sample of the new dataset structure:")
print(tokenized_datasets['train'])

🔤 Tokenizing datasets...


Map: 100%|██████████| 246/246 [00:00<00:00, 4678.09 examples/s]


✅ Datasets tokenized and formatted!
Sample of the new dataset structure:
Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 1960
})


In [12]:
# Cell 12 - Load the Pre-trained Model
from transformers import AutoModelForSequenceClassification
import torch

# Determine the device (use GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device} 🚀")

# Load the model, configured for our specific number of classes
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=num_classes
).to(device)

print(f"\n✅ Model '{MODEL_CKPT}' loaded successfully.")
print(f"Model configured for {num_classes} classes (poets).")



Using device: cuda 🚀


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google/muril-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



✅ Model 'google/muril-base-cased' loaded successfully.
Model configured for 41 classes (poets).


In [13]:
# Cell 13 - Define Metrics and Training Arguments for Early Stopping
from sklearn.metrics import accuracy_score, f1_score
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

# The compute_metrics function remains the same
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, predictions, average="macro")
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc, "f1": f1}

# Define the training arguments
training_args = TrainingArguments(
    output_dir="muril-base-poet-classifier-v2", # New output directory
    num_train_epochs=50,  # Set a high number, early stopping will halt it
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=3e-5,
    weight_decay=0.01,
    
    # --- Strategy for Early Stopping ---
    eval_strategy="steps",      # Evaluate at a fixed step interval
    eval_steps=250,                   # Evaluate every 250 steps (~once per epoch)
    logging_steps=250,
    save_steps=250,
    
    save_total_limit=2,               
    load_best_model_at_end=True,      # This is crucial for early stopping
    metric_for_best_model="f1",       
    fp16=True,                        
    push_to_hub=False,
)

In [14]:
# Cell 14 - Initialize Trainer with EarlyStoppingCallback

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)] # Add the callback here
)

# Start the training process!
print("🚀 Starting model training with Early Stopping...")
trainer.train()
print("✅ Training complete!")

🚀 Starting model training with Early Stopping...


/tmp/ipykernel_3514128/3043198377.py:4: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss,Validation Loss,Accuracy,F1
250,3.697600,3.657438,0.146939,0.070943
500,3.585100,3.499099,0.187755,0.077715
750,3.356800,3.281011,0.187755,0.082939
1000,3.074300,3.131812,0.200000,0.075163
1250,2.810400,2.911145,0.253061,0.122530
1500,2.572700,2.869974,0.236735,0.117022
1750,2.337100,2.712510,0.297959,0.162800
2000,2.096800,2.569320,0.314286,0.195652
2250,1.872600,2.527512,0.363265,0.262454
2500,1.623100,2.424079,0.371429,0.286146


✅ Training complete!
